# Descripción del dataset:

- `C`: color (1, light medium; 2, medium; 3, dark medium; 4, dark).
- `S`: spine condition (1, both good; 2, one worn or broken; 3, both worn or broken).
- `W`: carapace width, cm.
- `Wt`: weight, kg.
- `Sa`: number of satellites.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
display(df.head())
print(df.shape)

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(7, 5))
hb = plt.hexbin(df['W'], df['Sa'], gridsize=15, cmap='viridis', mincnt=1)
plt.colorbar(hb, label='Cantidad de puntos')
plt.title('Number of satellites by width of female crab')
plt.xlabel('Width (cm)')
plt.ylabel('Number of satellites')
plt.show()

In [ ]:
df['W_cat'] = pd.cut(
    df['W'],
    bins = [-1, 23.25, 24.25, 25.25, 26.25, 27.25, 28.25, 29.25, np.inf],
    labels = ['23.25>','23.25-24.25','24.25-25.25','25.25-26.25','26.25-27.25','27.25-28.25','28.25-29.25','29.25<']
)

cat_summary = df.groupby('W_cat').agg(
    n_obs = ('W_cat','count'),
    sum_satellites = ('Sa','sum'),
    mean_satellites = ('Sa','mean'),
    var_satellites = ('Sa','var'),
    mean_w = ('W','mean')
).round(2)

display(cat_summary)

In [ ]:
smooth_series = pd.DataFrame(lowess(df['Sa'], df['W'], frac=0.5), columns=['W','Sa'])
display(smooth_series.head())

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x='mean_w', y='mean_satellites', data=cat_summary, color='darkcyan',  label='Binned means')
sns.lineplot(x='W', y='Sa', data=smooth_series, label='LOWESS smoothing', linestyle='--', color='black')
plt.title('Smoothings of horseshoe crab counts')
plt.xlabel('Width (cm)')
plt.ylabel('Number of satellites')
plt.ylim(0,5.5)
plt.show()

In [ ]:
# Escalado de W
mean_W = df['W'].mean()
std_W = df['W'].std()
df['W_scaled'] = (df['W']-mean_W)/std_W

In [ ]:
model = smf.poisson(formula="Sa ~ W_scaled", data=df)
results = model.fit(disp=0)
print(results.summary())

In [ ]:
print((26.3 - mean_W) / std_W)
print((27.3 - mean_W) / std_W)



In [ ]:
results.predict(pd.DataFrame({'W_scaled':[0, 0.5]}))

### Modelo Binomial-Negativo para sobredispersión

In [ ]:
model_nb = smf.negativebinomial(formula="Sa ~ W_scaled", data=df)
results_nb = model_nb.fit(disp=0)

print(results_nb.summary())

In [ ]:
results_nb.predict(pd.DataFrame({'W_scaled':[0, 0.5]}))

In [ ]:
# Comparación rápida de Log-Likelihood
ll_poisson = results.llf  # Tu modelo previo
ll_nb = results_nb.llf

lr_stat = 2 * (ll_nb - ll_poisson)
print(f"Estadístico de la razón de verosimilitud: {lr_stat:.4f}")

### Comparación de modelos

In [ ]:
x_range = np.linspace(df['W_scaled'].min(), df['W_scaled'].max(), 100)
df_plot = pd.DataFrame({'W_scaled': x_range})

# 3. Obtener predicciones e intervalos de confianza (95%)
# Para el modelo Poisson (tu objeto 'results')
pred_poisson = results.get_prediction(df_plot).summary_frame(alpha=0.05)

# Para el modelo Binomial Negativa
pred_nb = results_nb.get_prediction(df_plot).summary_frame(alpha=0.05)

# 4. Visualización
plt.figure(figsize=(8, 6))

# Graficar los datos originales
plt.scatter(df['W_scaled'], df['Sa'], alpha=0.3, color='gray', label='Datos (Cangrejos)')

# Curva Poisson
plt.plot(x_range, pred_poisson['predicted'], color='red', lw=2, label='Tendencia Poisson')
plt.fill_between(x_range, pred_poisson['ci_lower'], pred_poisson['ci_upper'], 
                 color='red', alpha=0.1, label='IC 95% Poisson')

# Curva Binomial Negativa
plt.plot(x_range, pred_nb['predicted'], color='blue', lw=2, label='Tendencia Binomial Negativa')
plt.fill_between(x_range, pred_nb['ci_lower'], pred_nb['ci_upper'], 
                 color='blue', alpha=0.1, label='IC 95% Bin. Negativa')

plt.title('Comparativa: Poisson vs Binomial Negativa', fontsize=14)
plt.xlabel('Ancho de la Hembra (W_scaled)', fontsize=12)
plt.ylabel('Número de Satélites (Sa)', fontsize=12)
plt.legend()
plt.show()

# Comparación de Criterios de Información
print(f"AIC Poisson: {results.aic:.2f}")
print(f"AIC Binomial Negativa: {results_nb.aic:.2f}")